# Task B – MFCD vs VCD (Self-contained)
This notebook clones the repo, installs dependencies, downloads COCO val2014, runs VCD and MFCD, then prints a comparison table.

In [18]:
import torch
print('Torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA Version:', torch.version.cuda)

Torch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA Version: 12.8


In [2]:
import os
ROOT = '/content/VCD_project'
REPO_URL = 'https://github.com/maxheef/ECE209_Project1.git'
BRANCH = 'main'
COCO_VAL = '/content/datasets/coco/val2014'
MODEL_PATH = 'liuhaotian/llava-v1.5-7b'
SEED = 55

# MFCD parameters (from MFCD readme)
MFC_HIGH_ALPHA = 1.0
MFC_LOW_ALPHA = 1.0
MFC_BETA = 0.3
MFC_HIGH_PASS_CUTOFF = 0.1
MFC_LOW_PASS_CUTOFF = 0.9
MFC_FILTER_TYPE = 'gaussian'
print('ROOT=', ROOT)

ROOT= /content/VCD_project


In [25]:
%%bash
set -euo pipefail
cd /content
if [ -d /content/VCD_project/.git ]; then
  git -C /content/VCD_project fetch --depth 1 origin ${BRANCH:-main}
  git -C /content/VCD_project reset --hard FETCH_HEAD
else
  rm -rf /content/VCD_project
  git clone --depth 1 --branch ${BRANCH:-main} ${REPO_URL:-https://github.com/maxheef/ECE209_Project1.git} /content/VCD_project
fi

if [ ! -d /content/VCD_project/originalMFCD/mfcd ]; then
  echo 'Missing /content/VCD_project/originalMFCD/mfcd (check repo contents)'.
  find /content/VCD_project -maxdepth 3 -type d | sed -n '1,120p'
  exit 1
fi


HEAD is now at 9218012 TaskA VCD cell added to be self-contained


From https://github.com/maxheef/ECE209_Project1
 * branch            main       -> FETCH_HEAD
 + 54dd2df...9218012 main       -> origin/main  (forced update)


In [ ]:
import os
import subprocess
import shutil
from pathlib import Path

def run_cmd(cmd):
    process = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=os.environ.copy())
    for line in process.stdout:
        print(line, end='')
    process.wait()
    if process.returncode != 0:
        raise Exception(f"Command failed: {cmd}")

# --- Configuration ---
ENV_NAME = 'vcd310'
CONDA_PATH = '/usr/local/miniconda'
ENV_PATH = f'{CONDA_PATH}/envs/{ENV_NAME}'
PYTHON_BIN = f'{ENV_PATH}/bin/python'
PIP_BIN = f'{ENV_PATH}/bin/pip'
REQ_FILE = '/content/VCD_project/requirements.txt'

# 1) Clean and setup Miniconda
if os.path.exists(ENV_PATH):
    shutil.rmtree(ENV_PATH)

if not os.path.exists(CONDA_PATH):
    run_cmd('wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh')
    run_cmd(f'bash /tmp/miniconda.sh -b -p {CONDA_PATH}')

conda_bin = f'{CONDA_PATH}/bin/conda'

# 2) Accept ToS and create env
os.environ['CONDA_PLUGINS_AUTO_ACCEPT_TOS'] = 'yes'
run_cmd(f"{conda_bin} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main")
run_cmd(f"{conda_bin} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r")

print(f'Creating environment {ENV_NAME} (Python 3.10)...')
run_cmd(f"{conda_bin} create -n {ENV_NAME} python=3.10 -y")

# 3) Install requirements (torch/numpy/etc are in requirements.txt)
if not os.path.exists(REQ_FILE):
    raise FileNotFoundError(f'Missing requirements: {REQ_FILE}')
run_cmd(f"{PIP_BIN} install --upgrade pip setuptools wheel")
run_cmd(f"{PIP_BIN} install --no-cache-dir -r {REQ_FILE}")

# 4) Pin transformers to avoid llava config conflict + ensure numpy<2
run_cmd(f"{PIP_BIN} install --no-cache-dir 'transformers==4.35.2' 'numpy<2'")

# 5) Record python path for later cells
Path('/tmp/mytasks_python_bin.txt').write_text(PYTHON_BIN)
print('VCD env ready:', PYTHON_BIN)
run_cmd(f"{PYTHON_BIN} -c 'import torch; print(\"Python\", __import__(\"sys\").version); print(\"CUDA\", torch.cuda.is_available()); print(\"GPU\", torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"CPU\")'")


In [27]:
import os
import subprocess
import shutil

# --- Configuration ---
DATASET_DIR = "/content/datasets/coco"
VAL_DIR = os.path.join(DATASET_DIR, "val2014")
ZIP_FILE = os.path.join(DATASET_DIR, "val2014.zip")
ZIP_URL = "http://images.cocodataset.org/zips/val2014.zip"

# 1. Create directory structure
os.makedirs(DATASET_DIR, exist_ok=True)

# 2. Download and Unzip if val2014 doesn't exist
if not os.path.exists(VAL_DIR):
    print(f"Downloading COCO val2014 to {DATASET_DIR}...")
    # Using subprocess for wget/unzip to handle large file streams efficiently
    subprocess.run(["wget", "-q", ZIP_URL, "-O", ZIP_FILE], check=True)
    
    print("Unzipping dataset...this will take about 2 min")
    subprocess.run(["unzip", "-q", ZIP_FILE, "-d", DATASET_DIR], check=True)
    
    # Optional: Clean up zip to save space on Colab
    os.remove(ZIP_FILE)
    print("Download and extraction complete.")
else:
    print("COCO val2014 already exists. Skipping download.")

# 3. Verify: Print first 8 lines of directory listing
print("\n--- Directory Preview (val2014) ---")
files = sorted(os.listdir(VAL_DIR))
for f in files[:8]:
    print(f)


COCO val2014 already exists. Skipping download.

--- Directory Preview (val2014) ---
COCO_val2014_000000000042.jpg
COCO_val2014_000000000073.jpg
COCO_val2014_000000000074.jpg
COCO_val2014_000000000133.jpg
COCO_val2014_000000000136.jpg
COCO_val2014_000000000139.jpg
COCO_val2014_000000000143.jpg
COCO_val2014_000000000164.jpg


In [29]:
import os
import subprocess
from pathlib import Path

PYBIN_PATH = '/tmp/mytasks_python_bin.txt'
if not Path(PYBIN_PATH).exists():
    raise FileNotFoundError(f'Missing {PYBIN_PATH}. Run the conda setup cell first.')
PYBIN = Path(PYBIN_PATH).read_text().strip()

MODEL_PATH = 'liuhaotian/llava-v1.5-7b'
COCO_VAL = '/content/datasets/coco/val2014'
SEED = '55'
ROOT = '/content/VCD_project'
ORIG = f'{ROOT}/originalProject'

# Preflight checks for clearer errors
if not Path(ORIG, 'experiments').is_dir():
    raise FileNotFoundError(f'Missing originalProject/experiments at {ORIG}/experiments')
if not Path(COCO_VAL).is_dir():
    raise FileNotFoundError(f'Missing COCO val2014 at {COCO_VAL}')

env = os.environ.copy()
env['PYTHON_BIN'] = PYBIN
cmd = ['bash', '/content/VCD_project/myTasks/run_table1.sh', MODEL_PATH, COCO_VAL, SEED]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, env=env, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'run_table1.sh failed with code {result.returncode}')


Running: bash /content/VCD_project/myTasks/run_table1.sh liuhaotian/llava-v1.5-7b /content/datasets/coco/val2014 55
Running split=random method=regular

Traceback (most recent call last):
  File "/content/VCD_project/originalProject/experiments/./eval/object_hallucination_vqa_llava.py", line 12, in <module>
    from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
  File "/content/VCD_project/originalProject/experiments/llava/__init__.py", line 1, in <module>
    from .model import LlavaLlamaForCausalLM
  File "/content/VCD_project/originalProject/experiments/llava/model/__init__.py", line 1, in <module>
    from .language_model.llava_llama import LlavaLlamaForCausalLM, LlavaConfig
  File "/content/VCD_project/originalProject/experiments/llava/model/language_model/llava_llama.py", line 165, in <module>
    AutoConfig.register("llava", LlavaConfig)
  File "/usr/local/miniconda/envs/vcd310/lib/python3.10/site-packages/transformer

RuntimeError: run_table1.sh failed with code 1

In [ ]:
import subprocess
from pathlib import Path

PYBIN_PATH = '/tmp/mytasks_python_bin.txt'
if not Path(PYBIN_PATH).exists():
    raise FileNotFoundError(f'Missing {PYBIN_PATH}. Run the conda setup cell first.')
PYBIN = Path(PYBIN_PATH).read_text().strip()

mfcd_driver = 'import os\nimport json\nfrom pathlib import Path\nfrom PIL import Image\nimport torch\nfrom torch.utils.data import Dataset\n\nROOT = \'/content/VCD_project\'\nORIG = f\'{ROOT}/originalProject\'\nMFCD = f\'{ROOT}/originalMFCD/mfcd\'\nCOCO_VAL = \'/content/datasets/coco/val2014\'\nMODEL_PATH = \'liuhaotian/llava-v1.5-7b\'\nOUT_DIR = f\'{ROOT}/myTasks/output\'\n\nMFC_HIGH_ALPHA = 1.0\nMFC_LOW_ALPHA = 1.0\nMFC_BETA = 0.3\nMFC_HIGH_PASS_CUTOFF = 0.1\nMFC_LOW_PASS_CUTOFF = 0.9\nMFC_FILTER_TYPE = \'gaussian\'\n\nos.environ[\'PYTHONPATH\'] = MFCD + \':\' + os.environ.get(\'PYTHONPATH\',\'\')\n\nfrom eval_utils import prepare_for_generate, generate_responses, batch_prepare_data\nfrom eval_utils import MODEL_INFOS\nfrom eval.pope.eval import eval as pope_eval, recorder\n\nclass POPELocal(Dataset):\n    def __init__(self, json_path, image_root):\n        self.rows = [json.loads(line) for line in open(json_path, \'r\')]\n        self.image_root = Path(image_root)\n    def __len__(self):\n        return len(self.rows)\n    def __getitem__(self, idx):\n        row = self.rows[idx]\n        img = Image.open(self.image_root / row[\'image\']).convert(\'RGB\')\n        return {\n            \'question\': row[\'text\']\n            ,\'image\': img\n            ,\'label\': row[\'label\'].lower()\n        }\n\ndef run_mfcd(pope_type):\n    device = torch.device(\'cuda:0\' if torch.cuda.is_available() else \'cpu\')\n    class Args: pass\n    args = Args()\n    args.model_name_or_path = MODEL_PATH\n    args.model_type = \'llava-1.5\'\n    args.num_workers = 1\n    args.eval_batch_size = 1\n    args.device = \'cuda:0\'\n    args.temperature = 1.2\n    args.top_p = 1.0\n    args.top_k = 50\n    args.max_new_tokens = 2048\n    args.mfc_high_alpha = MFC_HIGH_ALPHA\n    args.mfc_low_alpha = MFC_LOW_ALPHA\n    args.mfc_beta = MFC_BETA\n    args.mfc_jsd = False\n    args.mfc_entropy = False\n    args.mfc_high_pass_cutoff = MFC_HIGH_PASS_CUTOFF\n    args.mfc_low_pass_cutoff = MFC_LOW_PASS_CUTOFF\n    args.mfc_filter_type = MFC_FILTER_TYPE\n\n    processor, generation_config, _ = prepare_for_generate(args=args, model_info=MODEL_INFOS[\'llava-1.5\'], device=device)\n\n    pope_json = f"{ORIG}/experiments/data/POPE/coco/coco_pope_{pope_type}.json"\n    dataset = POPELocal(pope_json, COCO_VAL)\n\n    prompts = batch_prepare_data(args=args, dataset=dataset, processor=processor, question_column_name=\'question\', image_column_name=\'image\')\n\n    model_info = MODEL_INFOS[\'llava-1.5\']\n    model = model_info.model_class.from_pretrained(MODEL_PATH, **model_info.model_config).to(device)\n\n    responses = generate_responses(model=model, processor=processor, prompts=prompts, generation_config=generation_config)\n\n    labels = [1 if d[\'label\'] == \'yes\' else 0 for d in dataset]\n    preds = recorder(responses)\n    return pope_eval(preds, labels)\n\nmfcd_random = run_mfcd(\'random\')\nmfcd_popular = run_mfcd(\'popular\')\nos.makedirs(OUT_DIR, exist_ok=True)\nwith open(os.path.join(OUT_DIR, \'mfcd_random.json\'), \'w\') as f:\n    json.dump(mfcd_random, f, indent=2)\nwith open(os.path.join(OUT_DIR, \'mfcd_popular.json\'), \'w\') as f:\n    json.dump(mfcd_popular, f, indent=2)\nprint(\'MFCD random:\', mfcd_random)\nprint(\'MFCD popular:\', mfcd_popular)'

subprocess.run([PYBIN, '-c', mfcd_driver], check=True)


In [ ]:
import json
import re
from pathlib import Path
P1 = '/content/VCD_project/myTasks'
SEED = 55
out_dir = Path(f'{P1}/output')
def parse_metrics_txt(path: Path):
    text = path.read_text()
    def g(name):
        m = re.search(rf"^{name}:\s*([0-9.]+)", text, re.MULTILINE)
        return float(m.group(1)) if m else float('nan')
    return {
        'accuracy': g('Accuracy'),
        'precision': g('Precision'),
        'recall': g('Recall'),
        'f1': g('F1'),
    }
def parse_metrics_json(path: Path):
    data = json.loads(path.read_text())
    return {
        'accuracy': float(data.get('accuracy', float('nan'))),
        'precision': float(data.get('precision', float('nan'))),
        'recall': float(data.get('recall', float('nan'))),
        'f1': float(data.get('f1', float('nan'))),
    }
vcd_random = parse_metrics_txt(out_dir / f'metrics_random_vcd_seed{SEED}.txt')
vcd_popular = parse_metrics_txt(out_dir / f'metrics_popular_vcd_seed{SEED}.txt')
mfcd_random = parse_metrics_json(out_dir / 'mfcd_random.json')
mfcd_popular = parse_metrics_json(out_dir / 'mfcd_popular.json')
print('| Split | Method | Accuracy | Precision | Recall | F1 |')
print('|---|---|---:|---:|---:|---:|')
print(f"| random | VCD | {vcd_random['accuracy']:.4f} | {vcd_random['precision']:.4f} | {vcd_random['recall']:.4f} | {vcd_random['f1']:.4f} |")
print(f"| random | MFCD | {mfcd_random['accuracy']:.4f} | {mfcd_random['precision']:.4f} | {mfcd_random['recall']:.4f} | {mfcd_random['f1']:.4f} |")
print(f"| popular | VCD | {vcd_popular['accuracy']:.4f} | {vcd_popular['precision']:.4f} | {vcd_popular['recall']:.4f} | {vcd_popular['f1']:.4f} |")
print(f"| popular | MFCD | {mfcd_popular['accuracy']:.4f} | {mfcd_popular['precision']:.4f} | {mfcd_popular['recall']:.4f} | {mfcd_popular['f1']:.4f} |")
